# 05 — Segment-Level Experiments

**Goal**: Demonstrate that zigzag TurnoverRate captures temporal dynamics that
generalise across mice, unlike raw spatial features.

| Experiment | Question | Method |
| --- | --- | --- |
| **A** | Can TurnoverRate distinguish segments within a video? | LOTO-CV segment classification |
| **B** | Is TurnoverRate consistent across trials for the same segment? | Within vs. between segment correlation |
| **C** | Does TurnoverRate generalise across mice? | Train mouse A → test mouse B |
| **C+** | Do mice share the same representational geometry? | Cross-mouse RSA on segment RDMs |

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
from scripts.run_segment_experiments import (
    build_segment_lookup, find_cross_mouse_pairs,
    compute_turnover_per_stim, extract_segment_turnover,
    SYNTHETIC_TYPES, OUT_DIR
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Segment Lookup Table

In [ ]:
seg_df = build_segment_lookup()
print(f'Total rows: {len(seg_df)}')
print(f'Mice: {seg_df["mouse"].nunique()},  Videos: {seg_df["video_id"].nunique()},  Unique segments: {seg_df["segment_id"].nunique()}')

for stype in SYNTHETIC_TYPES:
    sub = seg_df[seg_df['stim_type'] == stype]
    if len(sub):
        print(f'  {stype:>13s}: {sub["mouse"].nunique()} mice, '
              f'{sub["video_id"].nunique()} vids, '
              f'{sub["segment_id"].nunique()} unique segs, '
              f'{len(sub)} rows')

In [ ]:
pairs = find_cross_mouse_pairs(seg_df)
print(f'Cross-mouse video pairs: {len(pairs)}')
from collections import Counter
for k, v in sorted(Counter(s for _, _, s, _ in pairs).items()):
    print(f'  {k}: {v}')

## 2. Load Results

Load the pre-computed results from the SLURM run.
If results don't exist yet, submit the job with:
```bash
sbatch scripts/submit_segment.sh
```

In [ ]:
results_path = OUT_DIR / 'segment_results.json'
with open(results_path) as f:
    results = json.load(f)
print(f'Loaded results with keys: {list(results.keys())}')

## 3. Experiment A — Within-Video Segment Classification

In [ ]:
df_A = pd.DataFrame(results['experiment_A'])
print(f'Experiment A: {len(df_A)} (mouse, video) cases')
df_A.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: TurnoverRate F1 by stimulus type
ax = axes[0]
for stype in SYNTHETIC_TYPES:
    sub = df_A[df_A['stim_type'] == stype]
    if len(sub) == 0:
        continue
    ax.scatter(sub.index, sub['turnover_logreg_f1'], label=stype, s=40, alpha=0.7)
    ax.axhline(sub['chance'].mean(), ls='--', alpha=0.3)
ax.set_ylabel('Macro-F1')
ax.set_title('Exp A: Segment Classification (TurnoverRate LogReg)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)

# Right: Turnover vs Spatial comparison
ax = axes[1]
sub = df_A.dropna(subset=['spatial_logreg_f1'])
if len(sub) > 0:
    ax.scatter(sub['spatial_logreg_f1'], sub['turnover_logreg_f1'],
               c=[SYNTHETIC_TYPES.index(s) for s in sub['stim_type']],
               cmap='tab10', s=40, alpha=0.7)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('Spatial LogReg F1')
    ax.set_ylabel('TurnoverRate LogReg F1')
    ax.set_title('Segment: Turnover vs Spatial')
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../results/exp_A_segment_classification.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Experiment B — Cross-Trial Consistency

In [ ]:
df_B = pd.DataFrame(results['experiment_B'])
print(f'Experiment B: {len(df_B)} (mouse, video) cases')
df_B.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(df_B))
w = 0.35
ax.bar(x - w/2, df_B['within_corr_mean'], w, label='Within-segment', alpha=0.8)
ax.bar(x + w/2, df_B['between_corr_mean'], w, label='Between-segment', alpha=0.8)
ax.set_ylabel('Mean Pearson r')
ax.set_title('Exp B: Cross-Trial Segment Consistency')
ax.legend()
if len(df_B) <= 30:
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['mouse']}\n{r['stim_type'][:4]}" for _, r in df_B.iterrows()],
                        fontsize=6, rotation=45)
plt.tight_layout()
plt.savefig('../results/exp_B_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Experiment C — Cross-Mouse Generalization

In [ ]:
df_C = pd.DataFrame(results['experiment_C'])
print(f'Experiment C: {len(df_C)} cross-mouse pairs')
df_C.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: TurnoverRate cross-mouse F1 vs chance
ax = axes[0]
for stype in SYNTHETIC_TYPES:
    sub = df_C[df_C['stim_type'] == stype]
    if len(sub) == 0:
        continue
    ax.scatter(sub.index, sub['turnover_f1_mean'], label=stype, s=40, alpha=0.7)
ax.axhline(df_C['chance'].mean(), color='gray', ls='--', label='chance', alpha=0.5)
ax.set_ylabel('Macro-F1')
ax.set_title('Exp C: Cross-Mouse Segment Generalization (TurnoverRate)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)

# Right: Turnover vs Spatial in cross-mouse setting
ax = axes[1]
sub = df_C.dropna(subset=['spatial_f1_mean'])
if len(sub) > 0:
    ax.scatter(sub['spatial_f1_mean'], sub['turnover_f1_mean'],
               c=[SYNTHETIC_TYPES.index(s) for s in sub['stim_type']],
               cmap='tab10', s=40, alpha=0.7)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('Spatial F1 (cross-mouse)')
    ax.set_ylabel('TurnoverRate F1 (cross-mouse)')
    ax.set_title('Cross-Mouse: Turnover vs Spatial')
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../results/exp_C_cross_mouse.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Experiment C+ — RSA

In [ ]:
df_Cp = pd.DataFrame(results.get('experiment_C+', []))
if len(df_Cp) > 0:
    print(f'Experiment C+: {len(df_Cp)} RSA pairs')
    display(df_Cp.head(10))

    fig, ax = plt.subplots(figsize=(8, 5))
    for stype in SYNTHETIC_TYPES:
        sub = df_Cp[df_Cp['stim_type'] == stype]
        if len(sub) == 0:
            continue
        ax.scatter(sub.index, sub['rsa_spearman_r'], label=stype, s=50, alpha=0.7)
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_ylabel('Spearman r (RDM similarity)')
    ax.set_title('Exp C+: Cross-Mouse RSA (TurnoverRate)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('../results/exp_Cp_rsa.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No C+ results available yet.')

## 7. Summary Table

In [ ]:
summary_rows = []

# Experiment A summary
for stype in SYNTHETIC_TYPES:
    sub = df_A[df_A['stim_type'] == stype]
    if len(sub) == 0:
        continue
    row = {
        'Experiment': 'A (within-video)',
        'Stim Type': stype,
        'n': len(sub),
        'TurnoverRate F1': f"{sub['turnover_logreg_f1'].mean():.3f} ± {sub['turnover_logreg_f1'].std():.3f}",
        'Spatial F1': f"{sub['spatial_logreg_f1'].dropna().mean():.3f}" if sub['spatial_logreg_f1'].notna().any() else '—',
        'Chance': f"{sub['chance'].mean():.3f}",
    }
    summary_rows.append(row)

# Experiment C summary
if len(df_C) > 0:
    for stype in SYNTHETIC_TYPES:
        sub = df_C[df_C['stim_type'] == stype]
        if len(sub) == 0:
            continue
        row = {
            'Experiment': 'C (cross-mouse)',
            'Stim Type': stype,
            'n': len(sub),
            'TurnoverRate F1': f"{sub['turnover_f1_mean'].mean():.3f} ± {sub['turnover_f1_mean'].std():.3f}",
            'Spatial F1': f"{sub['spatial_f1_mean'].dropna().mean():.3f}" if sub['spatial_f1_mean'].notna().any() else '—',
            'Chance': f"{sub['chance'].mean():.3f}",
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))